
# **ML Modelling**

In [ ]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/type_data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/type_FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

In [ ]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# **Deep learning**








In [ ]:
# LM-BPNN
import numpy as np
import random
import os

from collections import Counter

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================================
# FIXED SEED (REPRODUCIBILITY)
# =========================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# =========================================
# SAFE STRATIFIED CV
# Automatically adapts to smallest class
# =========================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# =========================================
# LM-BPNN MODEL
# =========================================

lm_bpnn = ImbPipeline([

    # Standardization inside CV
    ("scaler", StandardScaler()),

    # Oversampling inside training folds only
    ("ros", RandomOverSampler(
        random_state=SEED
    )),

    # MLP approximation of LM-BPNN
    ("mlp", MLPClassifier(
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=SEED
    ))
])

# =========================================
# HYPERPARAMETER SEARCH SPACE
# =========================================

param_dist = {

    "mlp__hidden_layer_sizes": [
        (128, 64),
        (128, 128),
        (256, 128)
    ],

    "mlp__alpha": [
        0.0001,
        0.001,
        0.01
    ],

    "mlp__learning_rate_init": [
        0.001,
        0.005
    ]
}

# =========================================
# MAIN LOOP
# =========================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== LM-BPNN | FEATURE SET: {fs_name} ==========")

    # -------------------------------------
    # SELECT FEATURES
    # -------------------------------------

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # -------------------------------------
    # PREVENT RANDOMIZEDSEARCH WARNING
    # -------------------------------------

    total_space = int(np.prod([
        len(v) for v in param_dist.values()
    ]))

    n_iter = min(10, total_space)

    # -------------------------------------
    # RANDOM SEARCH
    # -------------------------------------

    search = RandomizedSearchCV(
        estimator=lm_bpnn,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,   # reproducible
        random_state=SEED
    )

    # -------------------------------------
    # TRAIN
    # -------------------------------------

    search.fit(X_train_sel, y_train)

    # -------------------------------------
    # TEST PREDICTION
    # -------------------------------------

    y_pred = search.predict(X_test_sel)

    # -------------------------------------
    # RESULTS
    # -------------------------------------

    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )